# Guía de uso — Docker para scripts `.py` y notebooks `.ipynb`

## Proyecto Big Data con PySpark

Esta guía explica cómo utilizar el entorno Docker del proyecto para trabajar con:

- scripts Python (`.py`);
- notebooks Jupyter (`.ipynb`);
- PySpark;
- archivos y datos del repositorio.

La idea central es que **Python, Java, Spark, Hadoop y Jupyter se ejecuten dentro de Docker**, mientras que el código y los datos permanecen guardados en el equipo.

```text
Windows 11
│
├── VS Code / navegador
├── Docker Desktop
└── repositorio local
      │
      │ bind mount
      ▼
Docker container
├── Python 3.11
├── Java 17
├── Spark / PySpark 4.1.3
├── Hadoop 3.4.2
└── JupyterLab
```

El repositorio se monta dentro del contenedor como:

```text
/workspace
```


# 1. Archivos Docker principales

```text
_project/
├── compose.yaml
├── docker/
│   ├── Dockerfile
│   ├── requirements.txt
│   └── .dockerignore
├── data/
├── notebooks/
├── src/
└── tests/
```

Responsabilidad:

```text
Dockerfile        → cómo construir el entorno
requirements.txt  → dependencias Python
compose.yaml      → cómo ejecutar el entorno
.dockerignore     → archivos excluidos del build
```


# 2. Preparación inicial

## Abrir Docker Desktop

Después de encender el equipo, abrir Docker Desktop y esperar a que el engine esté listo.

Comprobación opcional:

```cmd
docker run hello-world
```

Si aparece:

```text
Hello from Docker!
```

Docker está funcionando.

## Ir a la raíz del proyecto

En CMD:

```cmd
cd /d C:\Users\derec\Desktop\Proyectos\BigData\_project
```

## Construir la imagen

La primera vez, o cuando cambien dependencias:

```cmd
docker compose build
```

La imagen del proyecto es:

```text
bigdata-spark:0.1
```


# 3. Cuándo reconstruir la imagen

## Ejecutar `docker compose build` si cambia:

- `docker/Dockerfile`
- `docker/requirements.txt`

Ejemplos:

- nueva versión de Python;
- nueva versión de Java;
- nuevos paquetes Linux;
- nuevas librerías Python.

## No reconstruir si solo cambia:

```text
src/
tests/
notebooks/
data/
```

El bind mount hace visibles esos cambios automáticamente.

```text
¿Cambió Dockerfile o requirements.txt?
        │
        ├── Sí → docker compose build
        └── No → ejecutar directamente
```


# 4. Ejecutar archivos `.py`

Supongamos que existe:

```text
tests/smoke_test_spark.py
```

Desde la raíz del proyecto:

```cmd
docker compose run --rm spark python tests/smoke_test_spark.py
```

Desglose:

```text
docker compose run
        │
        ├── --rm     elimina el contenedor al terminar
        ├── spark    servicio definido en compose.yaml
        └── python tests/smoke_test_spark.py
```

Otros ejemplos:

```cmd
docker compose run --rm spark python src/etl.py
```

```cmd
docker compose run --rm spark python src/main.py
```


# 5. Entrar al contenedor con Bash

Cuando sea necesario inspeccionar el entorno:

```cmd
docker compose run --rm spark bash
```

Dentro del contenedor:

```bash
pwd
ls
python --version
java -version
```

Para salir:

```bash
exit
```


# 6. Ejecutar notebooks `.ipynb`

Para trabajar con notebooks, `docker/requirements.txt` debe incluir JupyterLab e ipykernel, además de PySpark.

Ejemplo:

```text
pyspark==4.1.3
jupyterlab
ipykernel
```

Si se agregaron después de construir la imagen:

```cmd
docker compose build
```

## Arrancar JupyterLab

Desde la raíz del proyecto:

```cmd
docker compose run --rm -p 8888:8888 spark jupyter lab --ip=0.0.0.0 --port=8888 --no-browser --allow-root --ServerApp.root_dir=/workspace
```

La terminal mostrará una URL similar a:

```text
http://127.0.0.1:8888/lab?token=...
```

Abrir esa dirección en el navegador.

Jupyter mostrará `/workspace`, que corresponde al repositorio local.


# 7. Verificar que el notebook usa Docker

Ejecutar en una celda:


In [ ]:
import sys

print("Python version:")
print(sys.version)

print("\nPython executable:")
print(sys.executable)


Se espera una ruta Linux, por ejemplo:

```text
/usr/local/bin/python
```

Eso confirma que el kernel está ejecutándose dentro del contenedor y no con el Python de Windows.


# 8. Crear una `SparkSession` desde el notebook


In [ ]:
from pyspark.sql import SparkSession

spark = (
    SparkSession.builder
    .master("local[*]")
    .appName("notebook-test")
    .getOrCreate()
)

print("Spark version:", spark.version)

spark.range(5).show()


`local[*]` indica que Spark se ejecuta localmente usando los recursos disponibles para el contenedor.


# 9. Verificar Java y Hadoop


In [ ]:
java_version = (
    spark.sparkContext
    ._jvm.java.lang.System
    .getProperty("java.version")
)

hadoop_version = (
    spark.sparkContext
    ._jvm.org.apache.hadoop.util.VersionInfo
    .getVersion()
)

print("Java:", java_version)
print("Hadoop:", hadoop_version)


En el entorno validado del proyecto se espera aproximadamente:

```text
Java: 17.x
Hadoop: 3.4.2
```


# 10. Leer y escribir datos

Dentro del contenedor:

```text
/workspace/data
```

corresponde a:

```text
_project/data
```

en Windows.

Ejemplo:


In [ ]:
output_path = "/workspace/data/example_parquet"

df = spark.range(10)

df.write \
    .mode("overwrite") \
    .parquet(output_path)

df_read = spark.read.parquet(output_path)

df_read.orderBy("id").show()


Los archivos creados por Spark aparecerán físicamente dentro de:

```text
_project/data/example_parquet/
```


# 11. Detener JupyterLab

Al terminar:

1. Guardar los notebooks.
2. Regresar a la terminal donde está Jupyter.
3. Presionar:

```text
Ctrl + C
```

4. Confirmar el cierre si Jupyter lo solicita.

El contenedor temporal se eliminará por `--rm`.

Los notebooks, scripts y datos permanecerán guardados porque viven en el repositorio local.


# 12. Rutina diaria recomendada

## Para archivos `.py`

```cmd
cd /d C:\Users\derec\Desktop\Proyectos\BigData\_project
```

```cmd
docker compose run --rm spark python ruta\al\script.py
```

Ejemplo:

```cmd
docker compose run --rm spark python tests\smoke_test_spark.py
```

## Para notebooks `.ipynb`

```cmd
cd /d C:\Users\derec\Desktop\Proyectos\BigData\_project
```

```cmd
docker compose run --rm -p 8888:8888 spark jupyter lab --ip=0.0.0.0 --port=8888 --no-browser --allow-root --ServerApp.root_dir=/workspace
```

Después:

1. abrir la URL mostrada;
2. entrar a `notebooks/`;
3. trabajar normalmente;
4. cerrar con `Ctrl + C`.

```text
PC encendida
    ↓
Docker Desktop
    ↓
CMD
    ↓
cd proyecto
    │
    ├── .py    → docker compose run --rm spark python ...
    │
    └── .ipynb → docker compose run --rm -p 8888:8888 spark jupyter lab ...
```


# 13. Comandos útiles

Ver Docker:

```cmd
docker --version
```

Ver contenedores activos:

```cmd
docker ps
```

Ver todos los contenedores:

```cmd
docker ps -a
```

Ver imágenes:

```cmd
docker images
```

Validar `compose.yaml`:

```cmd
docker compose config
```

Reconstruir imagen:

```cmd
docker compose build
```

Reconstruir sin caché:

```cmd
docker compose build --no-cache
```

Revisar espacio:

```cmd
docker system df
```


# 14. Problemas comunes

## Docker no responde

Ejemplo:

```text
failed to connect to the docker API
dockerDesktopLinuxEngine
```

Abrir Docker Desktop y esperar a que el engine inicie.

---

## `jupyter: command not found`

Revisar `docker/requirements.txt`.

Debe incluir:

```text
jupyterlab
ipykernel
```

Después:

```cmd
docker compose build
```

---

## PySpark no está disponible en el notebook

Probar:

```python
import pyspark
print(pyspark.__version__)
```

Si falla, revisar `requirements.txt` y reconstruir.

---

## Cambié `.py` o `.ipynb`

No reconstruir.

El bind mount hace visibles los cambios automáticamente.

---

## Puerto 8888 ocupado

Usar otro puerto del host:

```cmd
docker compose run --rm -p 8889:8888 spark jupyter lab --ip=0.0.0.0 --port=8888 --no-browser --allow-root --ServerApp.root_dir=/workspace
```

Luego abrir:

```text
http://localhost:8889
```


# 15. Idea principal

```text
DOCKER
    ↓
entorno reproducible
Python + Java + Spark + Hadoop + Jupyter

REPOSITORIO
    ↓
trabajo persistente
código + notebooks + datos + documentación
```

El contenedor puede destruirse y recrearse sin perder el trabajo.

El proyecto puede reproducirse en otro equipo siempre que existan:

```text
Docker
+
repositorio
+
Dockerfile
+
requirements.txt
+
compose.yaml
```
